# 07 — Städa nedladdade Pinterest-bilder
Kör ansiktsdetektion + mustasch-modellen på nedladdade bilder för att hitta skräp (collage, illustrationer, fel motiv, gruppbilder) innan manuell sortering. Generell — ändra bara `PINTEREST_DIR` nedan till vilken nedladdningsmapp du vill städa (t.ex. en specifik undermapp i `data/raw_downloads`, eller hela `data/raw_downloads` på en gång eftersom sökningen är rekursiv).

In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=178, margin=40, post_process=False)

mustache_model = tf.keras.models.load_model('models/mustache_detector_3.keras')

# Byt till vilken mapp du vill städa just nu — t.ex. en enskild sökmapp
# ('data/raw_downloads/stubble_mustache') eller hela raw_downloads på en gång
# (rekursiv sökning hittar allt i undermappar ändå).
PINTEREST_DIR = 'data/raw_downloads'

print('Redo!')

## Kör ansiktsdetektion + mustasch-modell på alla bilder

In [ ]:
files = []
for root, dirs, filenames in os.walk(PINTEREST_DIR):
    for fname in filenames:
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            files.append(os.path.join(root, fname))

print(f'Hittade {len(files)} filer')

results = []

for i, path in enumerate(files):
    fname = os.path.basename(path)
    try:
        img = Image.open(path).convert('RGB')
    except Exception:
        results.append((fname, path, None, 'kunde inte öppna bilden'))
        continue

    face = mtcnn(img)

    if face is None:
        results.append((fname, path, None, 'inget ansikte hittat'))
        continue

    arr = np.expand_dims(face.permute(1, 2, 0).numpy().astype(np.uint8), axis=0)
    mustache_prob = float(mustache_model.predict(arr, verbose=0)[0][0])
    results.append((fname, path, mustache_prob, 'ok'))

    if (i + 1) % 50 == 0:
        print(f'{i+1}/{len(files)} klara')

no_face     = [r for r in results if r[3] == 'inget ansikte hittat']
broken      = [r for r in results if r[3] == 'kunde inte öppna bilden']
low_mustache = sorted(
    [r for r in results if r[3] == 'ok' and r[2] < 0.3],
    key=lambda r: r[2]
)
good        = [r for r in results if r[3] == 'ok' and r[2] >= 0.3]

print(f'\nTotalt: {len(results)}')
print(f'Trasiga filer: {len(broken)}')
print(f'Inget ansikte hittat (troligen skräp): {len(no_face)}')
print(f'Ansikte men låg mustasch-confidence: {len(low_mustache)}')
print(f'Verkar OK (ansikte + mustasch): {len(good)}')


## Visa bilder utan ansikte — troligen collage/illustrationer/skräp

In [ ]:
def show_images(items, n=25, title='Bilder'):
    sample = items[:n]
    cols = 5
    rows = int(np.ceil(len(sample) / cols))

    plt.figure(figsize=(15, rows * 3))
    for i, item in enumerate(sample):
        fname, path, prob, status = item
        try:
            img = Image.open(path).convert('RGB')
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.axis('off')
            label = f'{prob:.2f}' if prob is not None else status
            plt.title(label, fontsize=9)
        except Exception:
            pass
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_images(no_face, n=25, title='Inget ansikte hittat')

## Visa bilder med ansikte men låg mustasch-confidence — granska manuellt

In [ ]:
show_images(low_mustache, n=25, title='Ansikte hittat men låg mustasch-confidence')

## Flytta skräp till en review-mapp
Granska bilderna ovan visuellt först — justera vilka grupper du faktiskt vill flytta.

In [ ]:
JUNK_DIR = 'data/pinterest_junk_review'
os.makedirs(JUNK_DIR, exist_ok=True)

to_move = no_face + broken

for fname, path, prob, status in to_move:
    dst = os.path.join(JUNK_DIR, fname)
    if os.path.exists(path):
        shutil.move(path, dst)


def count_files_recursive(folder):
    return sum(1 for _, _, fnames in os.walk(folder) for f in fnames
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))


print(f'Flyttade {len(to_move)} bilder till {JUNK_DIR}')
print(f'Kvar i {PINTEREST_DIR} (rekursivt, alla undermappar): {count_files_recursive(PINTEREST_DIR)}')

## Croppa till ansikten med MTCNN
Skriver över bilderna med MTCNN-croppade versioner — matchar exakt vad appen gör vid inferens. Kör EFTER skräpstädningen ovan.

In [ ]:
from facenet_pytorch import MTCNN

mtcnn_crop = MTCNN(image_size=178, margin=40, post_process=False)

remaining_files = [f for f in os.listdir(PINTEREST_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

cropped_count = 0
failed_count = 0

for fname in remaining_files:
    path = os.path.join(PINTEREST_DIR, fname)
    try:
        img = Image.open(path).convert('RGB')
    except Exception:
        continue

    face = mtcnn_crop(img)
    if face is None:
        failed_count += 1
        continue

    arr = face.permute(1, 2, 0).numpy().astype(np.uint8)
    cropped_img = Image.fromarray(arr)
    cropped_img.save(path)  # skriver över originalet med croppad version
    cropped_count += 1

print(f'Croppade och sparade: {cropped_count}')
print(f'Misslyckades (inget ansikte hittat denna gång): {failed_count}')

## Klart!
Bilderna är redan MTCNN-croppade av denna notebook (cellen ovan) — sortera dem **direkt** in i `data/epic_dataset/epic`, `medium` eller `thin` baserat på mustaschstil. Kör INTE `08`-croppningen på dem igen, de är redan i rätt format.